In [1]:
import asyncio
import sys
import threading
from playwright.async_api import async_playwright

_bg_loop = None
_bg_thread = None


def _ensure_bg_loop():
    """ProactorEventLoop를 백그라운드 스레드에서 실행"""
    global _bg_loop, _bg_thread
    if _bg_loop is None or not _bg_loop.is_running():
        if sys.platform == "win32":
            _bg_loop = asyncio.ProactorEventLoop()
        else:
            _bg_loop = asyncio.new_event_loop()
        _bg_thread = threading.Thread(target=_bg_loop.run_forever, daemon=True)
        _bg_thread.start()
    return _bg_loop


async def run_pw(coro):
    """Playwright 비동기 명령을 백그라운드 루프에서 실행
    사용법: result = await run_pw(page.goto('https://naver.com'))
    """
    loop = _ensure_bg_loop()
    future = asyncio.run_coroutine_threadsafe(coro, loop)
    while not future.done():
        await asyncio.sleep(0.05)
    return future.result()


print("Playwright 헬퍼 준비 완료")

Playwright 헬퍼 준비 완료


### PlayWright
- 자바스크립트에서 주로 데이터 수집할 때 사용하는 것을
- 파이썬에서 사용 가능하도록 한 패키지


### 2. 브라우저 띄우기
- 브라우저를 띄워서 제어 -> 브라우저는 작업동안애는 끄시면 x
- 작업이 다 끝나면 코드로 브라우저 종료시킬 예정

In [2]:
# 비동기모드로 실행하기
pw = await run_pw(async_playwright().start())

# 크롬 브라우저를 실행
browser = await run_pw(pw.chromium.launch(headless=False))

page = await run_pw(browser.new_page())

### 3. 페이지 접속

In [ ]:
await run_pw(page.goto("https://www.naver.com"))

# 대기코드
await run_pw(page.wait_for_timeout(2000))  




<Response url='https://www.naver.com/' request=<Request url='https://www.naver.com/' method='GET'>>

In [8]:
title = await run_pw(page.title())
print(title)

NAVER


In [11]:
search_box = page.get_by_placeholder("검색어를 입력해 주세요.")

# 검색어를 입력
await run_pw(search_box.fill("조승우"))
await run_pw(page.wait_for_timeout(2000))  



In [12]:
# 엔터를 쳐야 검색 가능
await run_pw(search_box.press("Enter"))
await run_pw(page.wait_for_timeout(2000))  

### 4.뉴스 검색

In [3]:
url = "https://search.naver.com/search.naver?where=news&query=조승우"
await run_pw(page.goto(url))
await run_pw(page.wait_for_timeout(2000))  

In [4]:
titles = page.locator('a[data-heatmap-target=".tit"]')

count = await run_pw(titles.count())
print(count)

19


In [5]:
# 내용도 추출
content = page.locator('a[data-heatmap-target=".body"]')

content_count = await run_pw(content.count())
print(content_count)


10


# 1. 제목을 찾는다.
# 2. 상위 태그를 찾는다.
# 3. 상위

In [6]:
# 한건 테스트
# 1. 제목을 찾는다.
title = titles.nth(0)


parent = title.locator("xpath=..")

# 3. 제목과 내용 쌍을 확인해보기
body = parent.locator('a[data-heatmap-target=".body"]')

has_body = await run_pw(body.count())
print(has_body)

1


In [7]:
total_data = [
    {
        "title" : "제목1",
        "content" : "내용1"
    },
    {
        "title" : "제목2",
        "content" : "내용2"
    }
]

In [8]:
# 총 title 과 content 를 for 문 을 이용해서 반복
total_data = []


for i in range(count):

    title = titles.nth(i)
    title_content = await run_pw(title.text_content())

    parent = title.locator("xpath=..")

    # 3. 제목과 내용 쌍을 확인해보기
    body = parent.locator('a[data-heatmap-target=".body"]')

    has_body = await run_pw(body.count())

    # 내용이 있을때는 내용을 가져오고 , 없을때는 "" 표시
    if has_body > 0:
        news_content = await run_pw(body.text_content())
    else:
        news_content = ""

    total_data.append(

        {
            "title" : title_content,
            "content" : news_content
        }

    )

    print(i,"번째 작업중입니다.")


0 번째 작업중입니다.
1 번째 작업중입니다.
2 번째 작업중입니다.
3 번째 작업중입니다.
4 번째 작업중입니다.
5 번째 작업중입니다.
6 번째 작업중입니다.
7 번째 작업중입니다.
8 번째 작업중입니다.
9 번째 작업중입니다.
10 번째 작업중입니다.
11 번째 작업중입니다.
12 번째 작업중입니다.
13 번째 작업중입니다.
14 번째 작업중입니다.
15 번째 작업중입니다.
16 번째 작업중입니다.
17 번째 작업중입니다.
18 번째 작업중입니다.


In [28]:
total_data

[{'title': '‘슈퍼 집돌이’ 조승우, 알고 보니 반전 성격 “촬영장 가면 I→E로 바...새 창 열림',
  'content': '배우 조승우가 ‘슈퍼 집돌이’다운 일상과 함께 일할 때만큼은 180도 달라지는 반전 성격을 공개했다. 31일 아레나 옴므 플러스 유튜브 채널에는 ‘왕으로서 예상치 못한 어려움을 겪은 조승우’라는 제목의 영상이 공개됐다. 이날 쉬는 날을 어떻게 보내느냐는 질문을 받은 조승우는 거의 집 밖으로 나가지... 새 창 열림'},
 {'title': '조승우 "애교·웃음 많은 편...집에선 존재감 없지만 현장 가면 \'E\'로 변...새 창 열림',
  'content': ''},
 {'title': "조승우→변요한, 20년 '타짜' 마지막 승부…추석 극장가 다시 접수새 창 열림",
  'content': "2006년 조승우의 '타짜'를 시작으로 추석마다 관객을 찾아온 시리즈가 올해 마지막 승부에 나선다. 이번에는 변요한과 노재원이 글로벌 도박판에서 친구에서 원수가 된 두 타짜로 맞붙는다. 영화 '타짜: 벨제붑의 노래'(감독 최국희)는 온라인 카지노 사업으로 세상을 다 가진 줄 알았던 장태영(변요한 분)과 그의... 새 창 열림"},
 {'title': '조승우, \'타짜4\' 변요한에 보낸 문자…"정 마담 김혜수 대사로 답장"새 창 열림',
  'content': '출연을 확정한 이후에는 \'타짜\' 1편에서 조승우가 맡았던 캐릭터와 관련해 직접 문자를 보내기도 했다. 이에 조승우는 정 마담의 대사인 "지나간 건 지나간 거야"라는 내용으로 답장을 보냈고, 새로운 작품인 만큼 재미있게 잘 만들어달라는 응원을 건넸다는 후문이다. 노재원은 친구이면서도 강한 질투심에... 새 창 열림'},
 {'title': '변요한, \'타짜4\' 출연 확정 후 조승우에 문자 "정마담 대사로 답장"[화보...새 창 열림',
  'content': ''},
 {'title': '조승우, \'타짜4\' 변요한에 보낸 문자 공개 "정 마담 김혜수 대사가.."

In [9]:
import pandas as pd 
df = pd.DataFrame(total_data)
df 

,title,content
0,"‘슈퍼 집돌이’ 조승우, 알고 보니 반전 성격 “촬영장 가면 I→E로 바...새 창 열림",배우 조승우가 ‘슈퍼 집돌이’다운 일상과 함께 일할 때만큼은 180도 달라지는 반전...
1,"조승우 ""애교·웃음 많은 편...집에선 존재감 없지만 현장 가면 'E'로 변...새...",
2,"조승우→변요한, 20년 '타짜' 마지막 승부…추석 극장가 다시 접수새 창 열림",2006년 조승우의 '타짜'를 시작으로 추석마다 관객을 찾아온 시리즈가 올해 마지막...
3,"조승우, '타짜4' 변요한에 보낸 문자…""정 마담 김혜수 대사로 답장""새 창 열림",출연을 확정한 이후에는 '타짜' 1편에서 조승우가 맡았던 캐릭터와 관련해 직접 문자...
4,"변요한, '타짜4' 출연 확정 후 조승우에 문자 ""정마담 대사로 답장""[화보...새...",
5,"조승우, '타짜4' 변요한에 보낸 문자 공개 ""정 마담 김혜수 대사가..""새 창 열림",
6,"변요한, ‘타짜: 벨제붑의 노래’ 출연 앞두고 조승우에 연락…돌아온 ...새 창 열림",
7,"'타짜4' 변요한·노재원 화보 공개 ""조승우 응원에 용기 얻었다""새 창 열림",
8,조승우 “청년도 중년도 아닌 위치…조금 더 숙성될 필요 있어” [화보...새 창 열림,배우 조승우의 화보가 공개됐다. 조승우가 매거진 ‘아레나 옴므 플러스’ 9월호의 표...
9,조승우 “‘동궁’ 어떻게 보면 비중 적지만..주저 없이 선택했죠”[화...새 창 열림,
